# SP-10: Model Validation with Hypothetical Scenarios

This notebook tests the v3 models with realistic poker scenarios to validate:
1. Models are sklearn Pipelines (no manual scaling needed)
2. Predictions are in reasonable ranges
3. Models respond correctly to different game situations

**IMPORTANT:** Uses EXACT feature lists from the trained models.

**Run this after SP-03, SP-05, and SP-09 to validate models are working correctly.**

In [ ]:
# Install dependencies
%pip install mlflow -q
dbutils.library.restartPython()

In [ ]:
# ============================================================================
# EXACT FEATURE LISTS FROM TRAINED MODELS
# ============================================================================

# 01-Opponent Model: 25 features (preflop), 28 features (postflop)
OPPONENT_FEATURES_PREFLOP = [
    'position_from_button', 'num_players', 'pot_size', 'amount',
    'action_no_in_hand', 'raises_so_far', 'calls_so_far',
    'starting_stack', 'stack_vs_table_median',
    'vpip_last3_hist', 'vpip_last5_hist', 'vpip_last10_hist',
    'pfr_last3_hist', 'pfr_last5_hist', 'pfr_last10_hist',
    'agg_factor_last3_hist', 'agg_factor_last5_hist', 'agg_factor_last10_hist',
    'street_adv_last3_hist', 'street_adv_last5_hist', 'street_adv_last10_hist',
    'stack_trend_last3_hist', 'stack_trend_last5_hist', 'stack_trend_last10_hist',
    'bet_pct_pot'
]

OPPONENT_FEATURES_POSTFLOP = OPPONENT_FEATURES_PREFLOP + [
    'board_pair_or_better', 'board_flush_possible', 'board_straight_possible'
]

# 02-Profit Model: 63 features (all streets) - CORRECTED!
PROFIT_FEATURES = [
    # Core features (20)
    'pot_size', 'hand_rank', 'starting_stack', 'stack_vs_table_median',
    'position_from_button', 'num_players', 'hand_equity',
    'board_pair_or_better', 'board_flush_possible', 'board_straight_possible',
    'hole_pair_flag', 'has_flush_draw_flag', 'has_straight_draw_flag',
    'bet_pct_pot', 'action_no_in_hand', 'raises_so_far', 'calls_so_far',
    'vpip_last3_hist', 'pfr_last3_hist', 'street_adv_last3_hist',
    # Additional historical stats (12)
    'agg_factor_last3_hist', 'stack_trend_last3_hist',
    'vpip_last5_hist', 'pfr_last5_hist', 'street_adv_last5_hist',
    'agg_factor_last5_hist', 'stack_trend_last5_hist',
    'vpip_last10_hist', 'pfr_last10_hist', 'street_adv_last10_hist',
    'agg_factor_last10_hist', 'stack_trend_last10_hist',
    # Table dynamics (6)
    'players_at_table', 'players_active', 'opponents_active',
    'avg_opponent_stack', 'max_opponent_stack', 'min_opponent_stack',
    # Position one-hot (8)
    'pos_count_BB', 'pos_count_BTN', 'pos_count_CO', 'pos_count_HJ',
    'pos_count_MP', 'pos_count_SB', 'pos_count_UTG', 'pos_count_UTG+1',
    # Board texture advanced (5)
    'board_monotone', 'board_paired', 'board_straighty',
    'board_flush_pressure', 'board_straight_pressure',
    # Opponent predictions (8)
    'predicted_strength', 'opponent_strength_mean', 'opponent_strength_max',
    'opponent_strength_min', 'opponent_air_count', 'opponent_middle_count',
    'opponent_nutted_count', 'has_opponent_predictions',
    # Pot/action context (4)
    'pot_before_action', 'facing_call', 'pot_odds_call', 'bb'
]

# 03-Policy Model: 44 features (all streets) - includes one-hot encoded columns
POLICY_FEATURES = [
    'starting_stack', 'stack_vs_table_median', 'position_from_button', 'num_players',
    'hand_equity',
    'board_pair_or_better', 'board_flush_possible', 'board_straight_possible',
    'hole_pair_flag', 'has_flush_draw_flag', 'has_straight_draw_flag',
    'vpip_last3_hist', 'pfr_last3_hist', 'agg_factor_last3_hist',
    'vpip_last5_hist', 'pfr_last5_hist', 'agg_factor_last5_hist',
    'vpip_last10_hist', 'pfr_last10_hist', 'agg_factor_last10_hist',
    'predicted_strength', 'opponent_strength_mean', 'opponent_strength_max',
    'opponent_strength_min', 'opponent_air_count', 'opponent_middle_count', 'opponent_nutted_count',
    'strength_advantage', 'strength_disadvantage', 'strength_strong',
    'heads_up', 'three_way', 'multiway',
    'position_bucket_CO', 'position_bucket_UTG+1', 'position_bucket_BTN',
    'position_bucket_UTG', 'position_bucket_BB', 'position_bucket_MP',
    'position_bucket_SB', 'position_bucket_HJ',
    'predicted_bucket_middle', 'predicted_bucket_air', 'predicted_bucket_nutted'
]

print(f"Opponent model preflop: {len(OPPONENT_FEATURES_PREFLOP)} features")
print(f"Opponent model postflop: {len(OPPONENT_FEATURES_POSTFLOP)} features")
print(f"Profit model: {len(PROFIT_FEATURES)} features")
print(f"Policy model: {len(POLICY_FEATURES)} features")

In [ ]:
# ============================================================================
# STEP 1: Load v3 Models from Unity Catalog
# ============================================================================
import mlflow
import mlflow.sklearn
import pandas as pd
from sklearn.pipeline import Pipeline

print("\n[1/6] Loading v3 models from MLflow Unity Catalog...")
print("   NOTE: Unity Catalog requires @alias syntax, not /latest")

MODEL_REGISTRY_PREFIX = "pokerml.default"
STREETS = ['preflop', 'flop', 'turn', 'river']

# Unity Catalog uses @champion alias (or @challenger, or version number like @1)
# Try multiple approaches: @champion first, then version 1
ALIAS_OPTIONS = ['@champion', '@1']

# Load all models
opponent_models = {}
profit_models = {}
policy_models = {}

def load_model_with_alias(model_base_name, alias_options):
    """Try loading a model with different alias options."""
    for alias in alias_options:
        model_uri = f"models:/{model_base_name}{alias}"
        try:
            model = mlflow.sklearn.load_model(model_uri)
            return model, model_uri, None
        except Exception as e:
            last_error = f"{type(e).__name__}: {str(e)[:100]}"
    return None, None, last_error

# Load Profit Models (REQUIRED for -504 BB diagnosis)
print("\n   --- Loading Profit Models (REQUIRED for -504 BB diagnosis) ---")
for street in STREETS:
    model_name = f"{MODEL_REGISTRY_PREFIX}.02-profit-modeling-{street}-v3"
    model, uri, error = load_model_with_alias(model_name, ALIAS_OPTIONS)
    if model:
        profit_models[street] = model
        print(f"   ✓ Loaded profit model for {street}")
        print(f"      URI: {uri}")
    else:
        print(f"   ✗ Failed to load profit model for {street}")
        print(f"      Tried: {model_name} with aliases {ALIAS_OPTIONS}")
        print(f"      Error: {error}")

print(f"\n   Profit models loaded: {list(profit_models.keys())}")

# Load Opponent Models
print("\n   --- Loading Opponent Models ---")
for street in STREETS:
    model_name = f"{MODEL_REGISTRY_PREFIX}.01-opponent-modeling-{street}-v3"
    model, uri, error = load_model_with_alias(model_name, ALIAS_OPTIONS)
    if model:
        opponent_models[street] = model
        print(f"   ✓ Loaded opponent model for {street}: {uri}")
    else:
        print(f"   ✗ Failed: {street} - {error}")

# Load Policy Models
print("\n   --- Loading Policy Models ---")
for street in STREETS:
    model_name = f"{MODEL_REGISTRY_PREFIX}.03-policy-modeling-{street}-v3"
    model, uri, error = load_model_with_alias(model_name, ALIAS_OPTIONS)
    if model:
        policy_models[street] = model
        print(f"   ✓ Loaded policy model for {street}: {uri}")
    else:
        print(f"   ✗ Failed: {street} - {error}")

# Summary
print("\n" + "=" * 60)
print("MODEL LOADING SUMMARY")
print("=" * 60)
print(f"   Opponent models: {list(opponent_models.keys()) or 'None'}")
print(f"   Profit models: {list(profit_models.keys()) or 'None'}")
print(f"   Policy models: {list(policy_models.keys()) or 'None'}")

if not profit_models:
    print("\n   ⚠️ NO PROFIT MODELS LOADED!")
    print("\n   To fix this, check what alias your models have in Unity Catalog:")
    print("   1. Go to Catalog > Models > pokerml.default.02-profit-modeling-preflop-v3")
    print("   2. Look at the 'Aliases' column - it might be 'champion', 'challenger', or none")
    print("   3. If no alias, add one or use version number like @1")
else:
    print(f"\n   ✓ Ready to diagnose -504 BB bug!")

In [ ]:
# ============================================================================
# STEP 2: Verify Models Are Pipelines with StandardScaler
# ============================================================================
print("\n[2/6] Verifying models are sklearn Pipelines...")

def verify_pipeline(model, name):
    """Check if model is a Pipeline with StandardScaler."""
    if isinstance(model, Pipeline):
        steps = [step[0] for step in model.steps]
        has_scaler = 'scaler' in steps
        print(f"   ✓ {name}: Pipeline with steps {steps}")
        if has_scaler:
            scaler = model.named_steps['scaler']
            n_features = len(scaler.mean_)
            print(f"      Expected features: {n_features}")
            print(f"      Scaler mean range: [{scaler.mean_.min():.2f}, {scaler.mean_.max():.2f}]")
            print(f"      Scaler std range: [{scaler.scale_.min():.2f}, {scaler.scale_.max():.2f}]")
        return True
    else:
        print(f"   ✗ {name}: NOT a Pipeline (type: {type(model)})")
        return False

print("\n   --- Opponent Models ---")
for street, model in opponent_models.items():
    verify_pipeline(model, f"opponent_{street}")

print("\n   --- Profit Models ---")
for street, model in profit_models.items():
    verify_pipeline(model, f"profit_{street}")

print("\n   --- Policy Models ---")
for street, model in policy_models.items():
    verify_pipeline(model, f"policy_{street}")

In [ ]:
# ============================================================================
# STEP 3: Test Opponent Model with Hypothetical Scenarios
# ============================================================================
import numpy as np  # Needed for np.integer check

print("\n[3/6] Testing Opponent Model predictions...")

# First, let's see what the model classes actually are
if opponent_models:
    sample_model = list(opponent_models.values())[0]
    if hasattr(sample_model, 'classes_'):
        print(f"   Model classes: {sample_model.classes_}")
    else:
        # For Pipeline, get the classifier's classes
        classifier = sample_model.named_steps.get('classifier') or sample_model.named_steps.get('model')
        if classifier and hasattr(classifier, 'classes_'):
            print(f"   Classifier classes: {classifier.classes_}")

# Class mapping (model may output 0, 1, 2 instead of strings)
CLASS_MAP = {0: 'air', 1: 'middle', 2: 'nutted'}

def get_class_name(cls):
    """Convert class to string name, handling both int and string classes."""
    if isinstance(cls, (int, float, np.integer)):
        return CLASS_MAP.get(int(cls), f'class_{int(cls)}')
    return str(cls)

def create_opponent_features(street, **overrides):
    """
    Create opponent model features with defaults.
    All chip values should be in BB!
    """
    defaults = {
        'position_from_button': 3,
        'num_players': 6,
        'pot_size': 5.0,           # 5 BB
        'amount': 2.5,             # 2.5 BB bet
        'action_no_in_hand': 2,
        'raises_so_far': 1,
        'calls_so_far': 0,
        'starting_stack': 100.0,   # 100 BB
        'stack_vs_table_median': 1.0,
        'vpip_last3_hist': 25,
        'vpip_last5_hist': 25,
        'vpip_last10_hist': 25,
        'pfr_last3_hist': 15,
        'pfr_last5_hist': 15,
        'pfr_last10_hist': 15,
        'agg_factor_last3_hist': 2.0,
        'agg_factor_last5_hist': 2.0,
        'agg_factor_last10_hist': 2.0,
        'street_adv_last3_hist': 50,
        'street_adv_last5_hist': 50,
        'street_adv_last10_hist': 50,
        'stack_trend_last3_hist': 0,
        'stack_trend_last5_hist': 0,
        'stack_trend_last10_hist': 0,
        'bet_pct_pot': 0.5,
        # Postflop only
        'board_pair_or_better': 0,
        'board_flush_possible': 0,
        'board_straight_possible': 0,
    }
    defaults.update(overrides)
    
    features = OPPONENT_FEATURES_PREFLOP if street == 'preflop' else OPPONENT_FEATURES_POSTFLOP
    return [defaults[f] for f in features], features

# Test scenarios
opponent_scenarios = [
    {
        'name': 'Tight player standard open (UTG)',
        'street': 'preflop',
        'overrides': {
            'position_from_button': 5,
            'pot_size': 1.5, 'amount': 2.5, 'bet_pct_pot': 1.67,
            'vpip_last10_hist': 15, 'pfr_last10_hist': 12,
            'agg_factor_last10_hist': 1.5
        },
        'expected': 'Likely middle or nutted (tight player opening UTG)'
    },
    {
        'name': 'LAG player 3-bet from SB',
        'street': 'preflop',
        'overrides': {
            'position_from_button': 1,
            'pot_size': 7.5, 'amount': 12, 'bet_pct_pot': 1.6,
            'raises_so_far': 2,
            'vpip_last10_hist': 40, 'pfr_last10_hist': 30,
            'agg_factor_last10_hist': 4.0
        },
        'expected': 'Could be anything (LAG 3-betting)'
    },
    {
        'name': 'Passive player river overbet',
        'street': 'river',
        'overrides': {
            'position_from_button': 0,
            'pot_size': 20, 'amount': 30, 'bet_pct_pot': 1.5,
            'raises_so_far': 0, 'calls_so_far': 3,
            'vpip_last10_hist': 35, 'pfr_last10_hist': 8,
            'agg_factor_last10_hist': 0.5,
            'board_flush_possible': 1
        },
        'expected': 'Likely nutted (passive player suddenly betting big)'
    },
]

print("\n" + "=" * 80)
print("OPPONENT MODEL PREDICTIONS")
print("=" * 80)

for scenario in opponent_scenarios:
    street = scenario['street']
    if street not in opponent_models:
        print(f"\n   Skipping {scenario['name']} - no {street} model")
        continue
    
    model = opponent_models[street]
    features, feature_names = create_opponent_features(street, **scenario['overrides'])
    
    X = pd.DataFrame([features], columns=feature_names)
    pred_class_raw = model.predict(X)[0]
    pred_proba = model.predict_proba(X)[0]
    
    # Get classes from model (handle Pipeline)
    if hasattr(model, 'classes_'):
        classes = model.classes_
    else:
        classifier = model.named_steps.get('classifier') or model.named_steps.get('model')
        classes = classifier.classes_ if classifier else [0, 1, 2]
    
    # Map numeric to string if needed
    pred_class = get_class_name(pred_class_raw)
    
    print(f"\n   Scenario: {scenario['name']}")
    print(f"   Street: {street}")
    print(f"   Expected: {scenario['expected']}")
    print(f"   \n   PREDICTION: {pred_class}")
    print(f"   Probabilities:")
    for i, prob in enumerate(pred_proba):
        cls_name = get_class_name(classes[i])
        bar = '█' * int(prob * 20)
        print(f"      {cls_name:8}: {prob:5.1%} {bar}")
    
    # Warning if model has wrong number of classes
    if len(classes) != 3:
        print(f"   ⚠️ WARNING: Model has {len(classes)} classes instead of 3!")
    print("-" * 60)

In [ ]:
# ============================================================================
# STEP 3b: COMPREHENSIVE OPPONENT MODEL TESTS - ALL STREETS
# ============================================================================
print("\n" + "=" * 80)
print("COMPREHENSIVE OPPONENT MODEL DIAGNOSTICS")
print("=" * 80)

# First, diagnose class distribution for each street's model
print("\n--- MODEL CLASS ANALYSIS ---")
for street in ['preflop', 'flop', 'turn', 'river']:
    if street not in opponent_models:
        print(f"\n   {street.upper()}: Model not loaded")
        continue
    
    model = opponent_models[street]
    
    # Get classifier from pipeline
    classifier = model.named_steps.get('classifier') or model.named_steps.get('model')
    if classifier and hasattr(classifier, 'classes_'):
        classes = classifier.classes_
        n_classes = len(classes)
        class_names = [get_class_name(c) for c in classes]
        print(f"\n   {street.upper()}: {n_classes} classes -> {class_names}")
        if n_classes < 3:
            print(f"   ⚠️ PROBLEM: Expected 3 classes (air, middle, nutted), got {n_classes}")
            print(f"   This means training data for {street} had no 'nutted' examples")
        else:
            print(f"   ✓ Correct number of classes")
    else:
        print(f"\n   {street.upper()}: Could not extract classes from model")

# ============================================================================
# EXTENDED TEST SCENARIOS - ALL STREETS
# ============================================================================
print("\n" + "=" * 80)
print("OPPONENT MODEL TESTS BY STREET")
print("=" * 80)

# Comprehensive scenarios for each street
all_street_scenarios = {
    'preflop': [
        {
            'name': 'Nit UTG open',
            'overrides': {
                'position_from_button': 5, 'pot_size': 1.5, 'amount': 2.5, 
                'bet_pct_pot': 1.67, 'raises_so_far': 1, 'calls_so_far': 0,
                'vpip_last10_hist': 12, 'pfr_last10_hist': 10, 'agg_factor_last10_hist': 1.2
            },
            'expected': 'middle or nutted (tight player = strong range)'
        },
        {
            'name': 'Fish limp-call',
            'overrides': {
                'position_from_button': 3, 'pot_size': 5.0, 'amount': 0, 
                'bet_pct_pot': 0, 'raises_so_far': 1, 'calls_so_far': 1,
                'vpip_last10_hist': 55, 'pfr_last10_hist': 5, 'agg_factor_last10_hist': 0.3
            },
            'expected': 'air or middle (loose passive = wide range)'
        },
        {
            'name': 'TAG 3-bet squeeze',
            'overrides': {
                'position_from_button': 0, 'pot_size': 12.0, 'amount': 12.0,
                'bet_pct_pot': 1.0, 'raises_so_far': 2, 'calls_so_far': 2,
                'vpip_last10_hist': 22, 'pfr_last10_hist': 18, 'agg_factor_last10_hist': 3.5
            },
            'expected': 'middle or nutted (3-bet squeeze = polarized)'
        },
        {
            'name': 'Maniac 4-bet shove',
            'overrides': {
                'position_from_button': 1, 'pot_size': 25.0, 'amount': 100.0,
                'bet_pct_pot': 4.0, 'raises_so_far': 3, 'calls_so_far': 0,
                'vpip_last10_hist': 48, 'pfr_last10_hist': 35, 'agg_factor_last10_hist': 5.0
            },
            'expected': 'could be anything (maniac = unpredictable)'
        },
    ],
    'flop': [
        {
            'name': 'C-bet on dry board',
            'overrides': {
                'position_from_button': 0, 'pot_size': 6.0, 'amount': 4.0,
                'bet_pct_pot': 0.67, 'raises_so_far': 0, 'calls_so_far': 0,
                'vpip_last10_hist': 25, 'pfr_last10_hist': 20, 'agg_factor_last10_hist': 2.5,
                'board_pair_or_better': 0, 'board_flush_possible': 0, 'board_straight_possible': 0
            },
            'expected': 'could be anything (standard c-bet)'
        },
        {
            'name': 'Check-raise on wet board',
            'overrides': {
                'position_from_button': 2, 'pot_size': 15.0, 'amount': 12.0,
                'bet_pct_pot': 0.8, 'raises_so_far': 1, 'calls_so_far': 1,
                'vpip_last10_hist': 28, 'pfr_last10_hist': 22, 'agg_factor_last10_hist': 2.0,
                'board_pair_or_better': 0, 'board_flush_possible': 1, 'board_straight_possible': 1
            },
            'expected': 'middle or nutted (check-raise = strong or draw)'
        },
        {
            'name': 'Donk bet from passive player',
            'overrides': {
                'position_from_button': 3, 'pot_size': 8.0, 'amount': 6.0,
                'bet_pct_pot': 0.75, 'raises_so_far': 0, 'calls_so_far': 0,
                'vpip_last10_hist': 40, 'pfr_last10_hist': 8, 'agg_factor_last10_hist': 0.4,
                'board_pair_or_better': 1, 'board_flush_possible': 0, 'board_straight_possible': 0
            },
            'expected': 'middle or nutted (passive player betting = has something)'
        },
        {
            'name': 'Overbet on monotone board',
            'overrides': {
                'position_from_button': 0, 'pot_size': 10.0, 'amount': 15.0,
                'bet_pct_pot': 1.5, 'raises_so_far': 0, 'calls_so_far': 0,
                'vpip_last10_hist': 30, 'pfr_last10_hist': 25, 'agg_factor_last10_hist': 3.0,
                'board_pair_or_better': 0, 'board_flush_possible': 1, 'board_straight_possible': 0
            },
            'expected': 'nutted or air (polarized overbet)'
        },
    ],
    'turn': [
        {
            'name': 'Double barrel on brick',
            'overrides': {
                'position_from_button': 0, 'pot_size': 15.0, 'amount': 10.0,
                'bet_pct_pot': 0.67, 'raises_so_far': 0, 'calls_so_far': 0,
                'action_no_in_hand': 5,
                'vpip_last10_hist': 25, 'pfr_last10_hist': 20, 'agg_factor_last10_hist': 2.5,
                'board_pair_or_better': 0, 'board_flush_possible': 0, 'board_straight_possible': 0
            },
            'expected': 'middle or nutted (continuing aggression)'
        },
        {
            'name': 'Check-call then lead turn',
            'overrides': {
                'position_from_button': 2, 'pot_size': 20.0, 'amount': 12.0,
                'bet_pct_pot': 0.6, 'raises_so_far': 0, 'calls_so_far': 1,
                'action_no_in_hand': 6,
                'vpip_last10_hist': 35, 'pfr_last10_hist': 12, 'agg_factor_last10_hist': 1.0,
                'board_pair_or_better': 0, 'board_flush_possible': 1, 'board_straight_possible': 0
            },
            'expected': 'middle or nutted (delayed aggression = strength)'
        },
        {
            'name': 'Turn raise vs double barrel',
            'overrides': {
                'position_from_button': 3, 'pot_size': 35.0, 'amount': 25.0,
                'bet_pct_pot': 0.71, 'raises_so_far': 1, 'calls_so_far': 0,
                'action_no_in_hand': 7,
                'vpip_last10_hist': 28, 'pfr_last10_hist': 22, 'agg_factor_last10_hist': 2.2,
                'board_pair_or_better': 0, 'board_flush_possible': 0, 'board_straight_possible': 1
            },
            'expected': 'nutted (turn raise = very strong)'
        },
        {
            'name': 'Small probe bet after check-check',
            'overrides': {
                'position_from_button': 0, 'pot_size': 8.0, 'amount': 3.0,
                'bet_pct_pot': 0.375, 'raises_so_far': 0, 'calls_so_far': 0,
                'action_no_in_hand': 4,
                'vpip_last10_hist': 30, 'pfr_last10_hist': 18, 'agg_factor_last10_hist': 1.8,
                'board_pair_or_better': 0, 'board_flush_possible': 0, 'board_straight_possible': 0
            },
            'expected': 'air or middle (small probe = thin value or bluff)'
        },
    ],
    'river': [
        {
            'name': 'Value bet on safe river',
            'overrides': {
                'position_from_button': 0, 'pot_size': 30.0, 'amount': 20.0,
                'bet_pct_pot': 0.67, 'raises_so_far': 0, 'calls_so_far': 0,
                'action_no_in_hand': 8,
                'vpip_last10_hist': 25, 'pfr_last10_hist': 20, 'agg_factor_last10_hist': 2.5,
                'board_pair_or_better': 0, 'board_flush_possible': 0, 'board_straight_possible': 0
            },
            'expected': 'middle or nutted (river value bet)'
        },
        {
            'name': 'River overbet shove',
            'overrides': {
                'position_from_button': 0, 'pot_size': 25.0, 'amount': 75.0,
                'bet_pct_pot': 3.0, 'raises_so_far': 0, 'calls_so_far': 0,
                'action_no_in_hand': 8,
                'vpip_last10_hist': 30, 'pfr_last10_hist': 25, 'agg_factor_last10_hist': 3.0,
                'board_pair_or_better': 0, 'board_flush_possible': 1, 'board_straight_possible': 0
            },
            'expected': 'nutted or air (polarized shove)'
        },
        {
            'name': 'Passive player river raise',
            'overrides': {
                'position_from_button': 2, 'pot_size': 50.0, 'amount': 40.0,
                'bet_pct_pot': 0.8, 'raises_so_far': 1, 'calls_so_far': 0,
                'action_no_in_hand': 9,
                'vpip_last10_hist': 38, 'pfr_last10_hist': 6, 'agg_factor_last10_hist': 0.4,
                'board_pair_or_better': 1, 'board_flush_possible': 0, 'board_straight_possible': 0
            },
            'expected': 'nutted (passive river raise = monster)'
        },
        {
            'name': 'Bluff catcher spot - small bet',
            'overrides': {
                'position_from_button': 0, 'pot_size': 40.0, 'amount': 12.0,
                'bet_pct_pot': 0.3, 'raises_so_far': 0, 'calls_so_far': 0,
                'action_no_in_hand': 8,
                'vpip_last10_hist': 32, 'pfr_last10_hist': 22, 'agg_factor_last10_hist': 2.0,
                'board_pair_or_better': 0, 'board_flush_possible': 0, 'board_straight_possible': 1
            },
            'expected': 'air or middle (small river bet = thin value or block)'
        },
    ],
}

# Run all tests
results_summary = {'pass': 0, 'fail': 0, 'skip': 0}

for street, scenarios in all_street_scenarios.items():
    print(f"\n{'='*80}")
    print(f"   {street.upper()} SCENARIOS")
    print(f"{'='*80}")
    
    if street not in opponent_models:
        print(f"   ⚠️ Skipped - {street} model not loaded")
        results_summary['skip'] += len(scenarios)
        continue
    
    model = opponent_models[street]
    
    # Get classifier classes
    classifier = model.named_steps.get('classifier') or model.named_steps.get('model')
    classes = classifier.classes_ if classifier and hasattr(classifier, 'classes_') else [0, 1, 2]
    n_classes = len(classes)
    
    # Show class warning once per street
    if n_classes < 3:
        print(f"   ⚠️ NOTE: This model only has {n_classes} classes: {[get_class_name(c) for c in classes]}")
        print(f"   (Missing 'nutted' class - training data issue)")
    
    for scenario in scenarios:
        features, feature_names = create_opponent_features(street, **scenario['overrides'])
        X = pd.DataFrame([features], columns=feature_names)
        
        pred_class_raw = model.predict(X)[0]
        pred_proba = model.predict_proba(X)[0]
        
        # Map prediction to string
        pred_class = get_class_name(pred_class_raw)
        
        # Check if prediction is reasonable
        max_prob = max(pred_proba)
        
        print(f"\n   {scenario['name']}")
        print(f"   Expected: {scenario['expected']}")
        print(f"   Predicted: {pred_class} ({max_prob:.1%} confidence)")
        
        # Show probability distribution
        proba_str = " | ".join([
            f"{get_class_name(c)}: {p:.1%}" 
            for c, p in zip(classes, pred_proba)
        ])
        print(f"   Distribution: {proba_str}")
        
        # Flag issues
        if n_classes < 3:
            print(f"   ⚠️ ISSUE: Only {n_classes} classes instead of 3")
            results_summary['fail'] += 1
        elif max_prob > 0.99:
            print(f"   ⚠️ ISSUE: 100% confidence suggests model not discriminating")
            results_summary['fail'] += 1
        else:
            print(f"   ✓ OK")
            results_summary['pass'] += 1

# Summary
print("\n" + "=" * 80)
print("OPPONENT MODEL TEST SUMMARY")
print("=" * 80)
print(f"   Passed: {results_summary['pass']}")
print(f"   Failed: {results_summary['fail']}")
print(f"   Skipped: {results_summary['skip']}")

if results_summary['fail'] > 0:
    print("\n   ⚠️ ISSUES DETECTED:")
    print("   - Preflop model only has 2 classes (air, middle) - missing 'nutted'")
    print("   - This is a TRAINING DATA issue in notebook 03")
    print("   - The preflop training data likely had no 'nutted' bucket labels")
    print("   - Check bucket_label distribution in SP-02 FeatureEngineering")

In [ ]:
# ============================================================================
# STEP 4: Test Profit Model with Hypothetical Scenarios
# ============================================================================
print("\n[4/6] Testing Profit Model predictions...")

# DEBUG: Show which profit models are available
print(f"\n   Available profit models: {list(profit_models.keys())}")
if not profit_models:
    print("   ⚠️ NO PROFIT MODELS LOADED!")
    print("   Check that models exist at: pokerml.default.02-profit-modeling-{street}-v3")

def create_profit_features(position_name='BTN', **overrides):
    """
    Create profit model features with defaults (63 features total).
    All chip values should be in BB!
    
    NOTE: hand_rank is 1-7462 where 1 is best (royal flush) and 7462 is worst.
    """
    defaults = {
        # Core features (20)
        'pot_size': 10.0,           # 10 BB
        'hand_rank': 3000,          # Middling hand (lower is better)
        'starting_stack': 100.0,    # 100 BB
        'stack_vs_table_median': 1.0,
        'position_from_button': 0,
        'num_players': 6,
        'hand_equity': 0.5,
        'board_pair_or_better': 0,
        'board_flush_possible': 0,
        'board_straight_possible': 0,
        'hole_pair_flag': 0,
        'has_flush_draw_flag': 0,
        'has_straight_draw_flag': 0,
        'bet_pct_pot': 0.5,
        'action_no_in_hand': 3,
        'raises_so_far': 1,
        'calls_so_far': 1,
        'vpip_last3_hist': 25,
        'pfr_last3_hist': 18,
        'street_adv_last3_hist': 50,
        # Additional historical stats (12)
        'agg_factor_last3_hist': 2.0,
        'stack_trend_last3_hist': 0,
        'vpip_last5_hist': 25,
        'pfr_last5_hist': 18,
        'street_adv_last5_hist': 50,
        'agg_factor_last5_hist': 2.0,
        'stack_trend_last5_hist': 0,
        'vpip_last10_hist': 25,
        'pfr_last10_hist': 18,
        'street_adv_last10_hist': 50,
        'agg_factor_last10_hist': 2.0,
        'stack_trend_last10_hist': 0,
        # Table dynamics (6)
        'players_at_table': 6,
        'players_active': 2,
        'opponents_active': 1,
        'avg_opponent_stack': 100.0,
        'max_opponent_stack': 120.0,
        'min_opponent_stack': 80.0,
        # Position one-hot (8)
        'pos_count_BB': 0, 'pos_count_BTN': 0, 'pos_count_CO': 0, 'pos_count_HJ': 0,
        'pos_count_MP': 0, 'pos_count_SB': 0, 'pos_count_UTG': 0, 'pos_count_UTG+1': 0,
        # Board texture advanced (5)
        'board_monotone': 0,
        'board_paired': 0,
        'board_straighty': 0,
        'board_flush_pressure': 0,
        'board_straight_pressure': 0,
        # Opponent predictions (8)
        'predicted_strength': 0.5,
        'opponent_strength_mean': 0.5,
        'opponent_strength_max': 0.5,
        'opponent_strength_min': 0.5,
        'opponent_air_count': 0,
        'opponent_middle_count': 1,
        'opponent_nutted_count': 0,
        'has_opponent_predictions': 1,
        # Pot/action context (4)
        'pot_before_action': 8.0,
        'facing_call': 5.0,
        'pot_odds_call': 0.38,
        'bb': 1.0,  # Big blind value (usually 1.0 when normalized)
    }
    
    # Apply overrides
    defaults.update(overrides)
    
    # Set position one-hot
    positions = ['BB', 'BTN', 'CO', 'HJ', 'MP', 'SB', 'UTG', 'UTG+1']
    for pos in positions:
        defaults[f'pos_count_{pos}'] = 1 if position_name == pos else 0
    
    return [defaults[f] for f in PROFIT_FEATURES]

# ============================================================================
# WEBAPP SCREENSHOT SCENARIO - 6d Jh preflop, 6 players
# ============================================================================

print("\n" + "=" * 80)
print("WEBAPP SCREENSHOT SCENARIO TESTS")
print("=" * 80)
print("\nReproducing the exact scenario from the webapp to diagnose the -504 BB bug")

# Define scenarios
webapp_scenario_correct = {
    'name': '✓ CORRECT: Webapp scenario (normalized to BB)',
    'position': 'BB',
    'overrides': {
        'hand_equity': 0.35,
        'hand_rank': 5500,
        'pot_size': 3.0,
        'starting_stack': 161.0,
        'pot_before_action': 3.0,
        'facing_call': 9.0,
        'pot_odds_call': 0.75,
        'bb': 1.0,
        'num_players': 6,
        'players_at_table': 6,
        'players_active': 6,
        'opponents_active': 5,
        'avg_opponent_stack': 197.0,
        'max_opponent_stack': 273.0,
        'min_opponent_stack': 157.0,
        'stack_vs_table_median': 0.82,
        'action_no_in_hand': 4,
        'raises_so_far': 1,
        'calls_so_far': 2,
        'bet_pct_pot': 3.67,
        'board_pair_or_better': 0,
        'board_flush_possible': 0,
        'board_straight_possible': 0,
        'board_monotone': 0,
        'board_paired': 0,
        'board_straighty': 0,
        'board_flush_pressure': 0,
        'board_straight_pressure': 0,
        'predicted_strength': 0.2,
        'opponent_strength_mean': 0.2,
        'opponent_strength_max': 0.2,
        'opponent_strength_min': 0.2,
        'opponent_air_count': 5,
        'opponent_middle_count': 0,
        'opponent_nutted_count': 0,
        'has_opponent_predictions': 1,
        'position_from_button': 2,
    },
    'expected_range': (-15, 5),
    'expected': 'Slightly negative (weak hand, facing raise)'
}

webapp_scenario_wrong = {
    'name': '⚠️ WRONG: If webapp passes CHIPS instead of BB',
    'position': 'BB',
    'overrides': {
        'hand_equity': 0.35,
        'hand_rank': 5500,
        'pot_size': 300.0,
        'starting_stack': 16100.0,
        'pot_before_action': 300.0,
        'facing_call': 900.0,
        'avg_opponent_stack': 19700.0,
        'max_opponent_stack': 27300.0,
        'min_opponent_stack': 15700.0,
        'bb': 100.0,
        'pot_odds_call': 0.75,
        'num_players': 6,
        'players_at_table': 6,
        'players_active': 6,
        'opponents_active': 5,
        'stack_vs_table_median': 0.82,
        'action_no_in_hand': 4,
        'raises_so_far': 1,
        'calls_so_far': 2,
        'bet_pct_pot': 3.67,
        'board_pair_or_better': 0,
        'board_flush_possible': 0,
        'board_straight_possible': 0,
        'board_monotone': 0,
        'board_paired': 0,
        'board_straighty': 0,
        'board_flush_pressure': 0,
        'board_straight_pressure': 0,
        'predicted_strength': 0.2,
        'opponent_strength_mean': 0.2,
        'opponent_strength_max': 0.2,
        'opponent_strength_min': 0.2,
        'opponent_air_count': 5,
        'opponent_middle_count': 0,
        'opponent_nutted_count': 0,
        'has_opponent_predictions': 1,
        'position_from_button': 2,
    },
    'expected_range': (-600, 600),
    'expected': 'GARBAGE - This is likely what the webapp is doing!'
}

webapp_scenario_strong = {
    'name': '✓ CORRECT: Same scenario but with AA',
    'position': 'BB',
    'overrides': {
        'hand_equity': 0.85,
        'hand_rank': 100,
        'pot_size': 3.0,
        'starting_stack': 161.0,
        'pot_before_action': 3.0,
        'facing_call': 9.0,
        'pot_odds_call': 0.75,
        'bb': 1.0,
        'num_players': 6,
        'players_at_table': 6,
        'players_active': 6,
        'opponents_active': 5,
        'avg_opponent_stack': 197.0,
        'max_opponent_stack': 273.0,
        'min_opponent_stack': 157.0,
        'stack_vs_table_median': 0.82,
        'action_no_in_hand': 4,
        'raises_so_far': 1,
        'calls_so_far': 2,
        'bet_pct_pot': 3.67,
        'board_pair_or_better': 0,
        'board_flush_possible': 0,
        'board_straight_possible': 0,
        'board_monotone': 0,
        'board_paired': 0,
        'board_straighty': 0,
        'board_flush_pressure': 0,
        'board_straight_pressure': 0,
        'predicted_strength': 0.2,
        'opponent_strength_mean': 0.2,
        'opponent_strength_max': 0.2,
        'opponent_strength_min': 0.2,
        'opponent_air_count': 5,
        'opponent_middle_count': 0,
        'opponent_nutted_count': 0,
        'has_opponent_predictions': 1,
        'position_from_button': 2,
    },
    'expected_range': (0, 20),
    'expected': 'Positive profit (premium hand vs weak opponents)'
}

profit_scenarios = [webapp_scenario_correct, webapp_scenario_wrong, webapp_scenario_strong]

print("\n" + "=" * 80)
print("PROFIT MODEL PREDICTIONS")
print("=" * 80)

# Find any available profit model
available_street = None
for street in ['preflop', 'flop', 'turn', 'river']:
    if street in profit_models:
        available_street = street
        break

if available_street is None:
    print("\n   ⚠️ NO PROFIT MODELS AVAILABLE!")
    print("   Please check that models are registered at:")
    print("   - pokerml.default.02-profit-modeling-preflop-v3")
    print("   - pokerml.default.02-profit-modeling-flop-v3")
    print("   etc.")
else:
    print(f"\n   Using profit model for street: {available_street}")
    model = profit_models[available_street]
    
    for scenario in profit_scenarios:
        features = create_profit_features(position_name=scenario.get('position', 'BTN'), **scenario['overrides'])
        
        X = pd.DataFrame([features], columns=PROFIT_FEATURES)
        pred_profit = model.predict(X)[0]
        
        low, high = scenario['expected_range']
        in_range = low <= pred_profit <= high
        
        print(f"\n{'='*60}")
        print(f"   Scenario: {scenario['name']}")
        print(f"{'='*60}")
        print(f"   Key inputs:")
        print(f"      pot_size: {scenario['overrides'].get('pot_size', 10)}")
        print(f"      starting_stack: {scenario['overrides'].get('starting_stack', 100)}")
        print(f"      facing_call: {scenario['overrides'].get('facing_call', 0)}")
        print(f"      hand_equity: {scenario['overrides'].get('hand_equity', 0.5)}")
        print(f"   \n   PREDICTED PROFIT: {pred_profit:.2f} BB")
        print(f"   Expected: {scenario['expected']}")
        print(f"   Expected range: [{low}, {high}] BB")
        
        if 'WRONG' in scenario['name']:
            if abs(pred_profit) > 50:
                print(f"\n   ⚠️  CONFIRMED BUG: Garbage output ({pred_profit:.1f} BB)")
                print(f"   ⚠️  This matches the webapp's -504 BB prediction!")
                print(f"   ⚠️  CAUSE: Webapp is passing CHIP values, not BB values!")
            else:
                print(f"   ✓ Model handled extreme values gracefully")
        elif in_range:
            print(f"   ✓ Prediction in expected range")
        else:
            print(f"   ⚠️ Prediction outside expected range!")

print("\n" + "=" * 80)
print("DIAGNOSIS")
print("=" * 80)
print("""
If the 'WRONG' scenario produces output similar to -504 BB, then:

THE BUG IS IN THE WEBAPP:
   The webapp is passing chip values instead of BB-normalized values.
   
FIX REQUIRED IN WEBAPP:
   Before calling the profit model, divide all chip amounts by big_blind:
   
   pot_size_bb = pot_chips / big_blind
   starting_stack_bb = stack_chips / big_blind
   facing_call_bb = call_chips / big_blind
   pot_before_action_bb = pot_chips / big_blind
   avg_opponent_stack_bb = avg_stack_chips / big_blind
   etc.
""")

In [ ]:
# ============================================================================
# DIAGNOSTIC: Investigate Profit Model Issue
# ============================================================================
print("\n" + "=" * 80)
print("PROFIT MODEL DIAGNOSTIC")
print("=" * 80)

# Get the preflop profit model
profit_model = profit_models.get('preflop')
if not profit_model:
    print("No preflop profit model available!")
else:
    print("\n1. MODEL STRUCTURE:")
    print(f"   Type: {type(profit_model)}")
    if hasattr(profit_model, 'steps'):
        print(f"   Pipeline steps: {[step[0] for step in profit_model.steps]}")
    
    # Get the scaler
    scaler = profit_model.named_steps.get('scaler')
    if scaler:
        print(f"\n2. SCALER STATISTICS (what the model was trained on):")
        print(f"   Number of features: {len(scaler.mean_)}")
        print(f"\n   Feature means (first 20):")
        for i, (feat, mean, scale) in enumerate(zip(PROFIT_FEATURES[:20], scaler.mean_[:20], scaler.scale_[:20])):
            print(f"      {feat:30s}: mean={mean:12.2f}, std={scale:12.2f}")
        
        print(f"\n   Feature means (stack-related):")
        stack_features = ['pot_size', 'starting_stack', 'facing_call', 'avg_opponent_stack', 
                         'max_opponent_stack', 'min_opponent_stack', 'pot_before_action']
        for feat in stack_features:
            if feat in PROFIT_FEATURES:
                idx = PROFIT_FEATURES.index(feat)
                print(f"      {feat:25s}: mean={scaler.mean_[idx]:12.2f}, std={scaler.scale_[idx]:12.2f}")
    
    # Get the regressor
    regressor = profit_model.named_steps.get('regressor') or profit_model.named_steps.get('model')
    if regressor:
        print(f"\n3. REGRESSOR:")
        print(f"   Type: {type(regressor).__name__}")
        if hasattr(regressor, 'feature_importances_'):
            importances = regressor.feature_importances_
            top_features = sorted(zip(PROFIT_FEATURES, importances), key=lambda x: -x[1])[:10]
            print(f"\n   Top 10 most important features:")
            for feat, imp in top_features:
                print(f"      {feat:30s}: {imp:.4f}")

    # Test with values closer to training data means
    print(f"\n4. TEST WITH TRAINING-LIKE VALUES:")
    if scaler:
        # Use the mean values from training
        test_features = list(scaler.mean_)
        X_test = pd.DataFrame([test_features], columns=PROFIT_FEATURES)
        pred = profit_model.predict(X_test)[0]
        print(f"   Prediction with MEAN training values: {pred:.2f} BB")
        print(f"   (This should be close to 0 if model is well-calibrated)")

    # Check if target was in BB or chips during training
    print(f"\n5. LIKELY ROOT CAUSE:")
    if scaler:
        pot_mean = scaler.mean_[PROFIT_FEATURES.index('pot_size')]
        stack_mean = scaler.mean_[PROFIT_FEATURES.index('starting_stack')]
        
        if pot_mean > 100:
            print(f"   ⚠️ pot_size mean = {pot_mean:.0f} - MODEL WAS TRAINED ON CHIP VALUES!")
            print(f"   ⚠️ starting_stack mean = {stack_mean:.0f}")
            print(f"\n   The model expects CHIP values, not BB values!")
            print(f"   Either retrain with BB-normalized data, or pass chips to the model.")
        elif pot_mean < 50:
            print(f"   ✓ pot_size mean = {pot_mean:.1f} - Model was trained on BB values")
            print(f"   ✓ starting_stack mean = {stack_mean:.1f}")
            print(f"\n   But predictions are still wrong - check target variable normalization")

In [ ]:
# ============================================================================
# TEST: Use values matching training data distribution
# ============================================================================
print("\n" + "=" * 80)
print("TESTING WITH TRAINING-LIKE VALUES")
print("=" * 80)

profit_model = profit_models.get('preflop')
scaler = profit_model.named_steps.get('scaler')

# Create test scenarios using values CLOSE to training means
print("\n⚠️ KEY FINDING: hand_rank mean = 940 MILLION (should be 1-7462)")
print("   This indicates the training data had corrupted hand_rank values!")

# Test 1: Use exact training means
print("\n1. EXACT TRAINING MEANS:")
test_means = list(scaler.mean_)
X_means = pd.DataFrame([test_means], columns=PROFIT_FEATURES)
pred_means = profit_model.predict(X_means)[0]
print(f"   Prediction: {pred_means:.2f} BB")

# Test 2: Adjust only hand_rank to realistic value
print("\n2. FIX hand_rank TO REALISTIC VALUE (5000 = weak hand):")
test_fixed = list(scaler.mean_)
hand_rank_idx = PROFIT_FEATURES.index('hand_rank')
test_fixed[hand_rank_idx] = 5000  # Realistic weak hand
X_fixed = pd.DataFrame([test_fixed], columns=PROFIT_FEATURES)
pred_fixed = profit_model.predict(X_fixed)[0]
print(f"   Prediction: {pred_fixed:.2f} BB")

# Test 3: Use the corrupted hand_rank value the model expects
print("\n3. USE CORRUPTED hand_rank (matching training data):")
test_corrupted = list(scaler.mean_)
test_corrupted[hand_rank_idx] = 940175568  # Match training mean
X_corrupted = pd.DataFrame([test_corrupted], columns=PROFIT_FEATURES)
pred_corrupted = profit_model.predict(X_corrupted)[0]
print(f"   Prediction: {pred_corrupted:.2f} BB")

# Test 4: Webapp scenario with corrupted hand_rank
print("\n4. WEBAPP SCENARIO WITH CORRUPTED hand_rank:")
webapp_features = create_profit_features(position_name='BB', **{
    'hand_equity': 0.35,
    'hand_rank': 940175568,  # Use the corrupted scale!
    'pot_size': 55,          # Match training mean
    'starting_stack': 413,   # Match training mean
    'facing_call': 14,       # Match training mean
    'avg_opponent_stack': 2791,  # Match training mean
    'max_opponent_stack': 758,
    'min_opponent_stack': 165,
    'pot_before_action': 46,
    'bb': 1.0,
})
X_webapp = pd.DataFrame([webapp_features], columns=PROFIT_FEATURES)
pred_webapp = profit_model.predict(X_webapp)[0]
print(f"   Prediction: {pred_webapp:.2f} BB")

print("\n" + "=" * 80)
print("DIAGNOSIS SUMMARY")
print("=" * 80)
print("""
THE PROFIT MODEL HAS A CRITICAL DATA ISSUE:

1. hand_rank was trained with values ~940 MILLION instead of 1-7462
   - This could be: hand_rank * some_id, wrong column, or data corruption
   
2. Stack values are inconsistent:
   - avg_opponent_stack mean = 2791 (likely chips, not BB)
   - starting_stack mean = 413 (could be BB for deep stack)
   
RECOMMENDED FIX:
   The model needs to be RETRAINED with correct data:
   
   1. Fix hand_rank calculation in feature engineering (SP-03)
   2. Ensure ALL chip values are BB-normalized consistently
   3. Retrain the profit model (SP-05)
   
   Check notebook 03-FeatureEngineering to find where hand_rank is computed.
""")

In [ ]:
# ============================================================================
# STEP 5: Test Policy Model with Hypothetical Scenarios
# ============================================================================
print("\n[5/6] Testing Policy Model predictions...")

# Policy class mapping (model outputs integers, we want strings)
POLICY_CLASS_MAP = {0: 'bet', 1: 'call', 2: 'fold'}

def get_policy_class_name(cls):
    """Convert policy class to string name."""
    if isinstance(cls, (int, float, np.integer)):
        return POLICY_CLASS_MAP.get(int(cls), f'class_{int(cls)}')
    return str(cls)

def create_policy_features(position_name='BTN', predicted_bucket='middle', **overrides):
    """
    Create policy model features with defaults.
    Includes one-hot encoded position and bucket columns.
    """
    defaults = {
        'starting_stack': 100.0,
        'stack_vs_table_median': 1.0,
        'position_from_button': 0,
        'num_players': 6,
        'hand_equity': 0.5,
        'board_pair_or_better': 0,
        'board_flush_possible': 0,
        'board_straight_possible': 0,
        'hole_pair_flag': 0,
        'has_flush_draw_flag': 0,
        'has_straight_draw_flag': 0,
        'vpip_last3_hist': 25, 'pfr_last3_hist': 18, 'agg_factor_last3_hist': 2.0,
        'vpip_last5_hist': 25, 'pfr_last5_hist': 18, 'agg_factor_last5_hist': 2.0,
        'vpip_last10_hist': 25, 'pfr_last10_hist': 18, 'agg_factor_last10_hist': 2.0,
        'predicted_strength': 0.5,
        'opponent_strength_mean': 0.5,
        'opponent_strength_max': 0.5,
        'opponent_strength_min': 0.5,
        'opponent_air_count': 0,
        'opponent_middle_count': 1,
        'opponent_nutted_count': 0,
        'strength_advantage': 0.0,
        'strength_disadvantage': 0,
        'strength_strong': 0,
        'heads_up': 1,
        'three_way': 0,
        'multiway': 0,
        # One-hot position (will be set based on position_name)
        'position_bucket_CO': 0, 'position_bucket_UTG+1': 0, 'position_bucket_BTN': 0,
        'position_bucket_UTG': 0, 'position_bucket_BB': 0, 'position_bucket_MP': 0,
        'position_bucket_SB': 0, 'position_bucket_HJ': 0,
        # One-hot predicted bucket (will be set based on predicted_bucket)
        'predicted_bucket_middle': 0, 'predicted_bucket_air': 0, 'predicted_bucket_nutted': 0,
    }
    
    # Apply overrides
    defaults.update(overrides)
    
    # Set one-hot position
    positions = ['CO', 'UTG+1', 'BTN', 'UTG', 'BB', 'MP', 'SB', 'HJ']
    for pos in positions:
        defaults[f'position_bucket_{pos}'] = 1 if position_name == pos else 0
    
    # Set one-hot predicted bucket
    buckets = ['middle', 'air', 'nutted']
    for bucket in buckets:
        defaults[f'predicted_bucket_{bucket}'] = 1 if predicted_bucket == bucket else 0
    
    return [defaults[f] for f in POLICY_FEATURES]

# Test scenarios
policy_scenarios = [
    {
        'name': 'Monster hand on button',
        'position': 'BTN',
        'predicted_bucket': 'air',
        'overrides': {
            'hand_equity': 0.95,
            'position_from_button': 0,
            'strength_advantage': 0.45,
            'strength_strong': 1,
            'opponent_strength_mean': 0.5,
            'opponent_air_count': 1,
        },
        'expected': 'bet (value betting monster)'
    },
    {
        'name': 'Weak hand out of position vs nutted opponent',
        'position': 'UTG',
        'predicted_bucket': 'nutted',
        'overrides': {
            'hand_equity': 0.15,
            'position_from_button': 5,
            'strength_advantage': -0.65,
            'strength_disadvantage': 1,
            'opponent_strength_mean': 0.8,
            'opponent_nutted_count': 1,
        },
        'expected': 'fold (weak vs strong)'
    },
    {
        'name': 'Flush draw in position heads up',
        'position': 'BTN',
        'predicted_bucket': 'middle',
        'overrides': {
            'hand_equity': 0.35,
            'has_flush_draw_flag': 1,
            'board_flush_possible': 1,
            'strength_advantage': -0.15,
            'heads_up': 1,
        },
        'expected': 'call or bet (drawing hand with position)'
    },
    {
        'name': 'Premium pair multiway',
        'position': 'CO',
        'predicted_bucket': 'middle',
        'overrides': {
            'hand_equity': 0.75,
            'hole_pair_flag': 1,
            'strength_advantage': 0.25,
            'strength_strong': 1,
            'heads_up': 0,
            'multiway': 1,
            'opponent_middle_count': 2,
        },
        'expected': 'bet (protect premium multiway)'
    },
]

print("\n" + "=" * 80)
print("POLICY MODEL PREDICTIONS")
print("=" * 80)

# First check what classes the model has
if policy_models:
    sample_model = list(policy_models.values())[0]
    classifier = sample_model.named_steps.get('classifier') or sample_model.named_steps.get('model')
    if classifier and hasattr(classifier, 'classes_'):
        print(f"\n   Model classes: {[get_policy_class_name(c) for c in classifier.classes_]}")

for scenario in policy_scenarios:
    street = 'flop'  # Use flop model for testing
    if street not in policy_models:
        print(f"   No policy model for {street}")
        continue
    
    model = policy_models[street]
    features = create_policy_features(
        position_name=scenario['position'],
        predicted_bucket=scenario['predicted_bucket'],
        **scenario['overrides']
    )
    
    X = pd.DataFrame([features], columns=POLICY_FEATURES)
    
    pred_action_raw = model.predict(X)[0]
    pred_action = get_policy_class_name(pred_action_raw)
    pred_proba = model.predict_proba(X)[0]
    
    # Get classes from model
    classifier = model.named_steps.get('classifier') or model.named_steps.get('model')
    classes = classifier.classes_ if classifier and hasattr(classifier, 'classes_') else [0, 1, 2]
    
    # Calculate confidence
    sorted_probs = sorted(pred_proba, reverse=True)
    confidence = sorted_probs[0] - sorted_probs[1]
    
    print(f"\n   Scenario: {scenario['name']}")
    print(f"   Position: {scenario['position']}")
    print(f"   Opponent read: {scenario['predicted_bucket']}")
    print(f"   Hand equity: {scenario['overrides'].get('hand_equity', 0.5)}")
    print(f"   Strength advantage: {scenario['overrides'].get('strength_advantage', 0)}")
    print(f"   ")
    print(f"   RECOMMENDED ACTION: {pred_action}")
    print(f"   Confidence: {confidence:.1%}")
    print(f"   Expected: {scenario['expected']}")
    print(f"   Action probabilities:")
    for cls, prob in sorted(zip(classes, pred_proba), key=lambda x: -x[1]):
        cls_name = get_policy_class_name(cls)
        bar = '#' * int(prob * 20)
        print(f"      {cls_name:10}: {prob:5.1%} {bar}")
    print("-" * 60)

In [ ]:
# ============================================================================
# OBVIOUS BET/RAISE SCENARIOS - DIAGNOSTIC TEST
# ============================================================================
# These scenarios should OBVIOUSLY recommend 'bet' - if they don't, the model has issues
print("\n" + "=" * 80)
print("OBVIOUS BET/RAISE SCENARIOS - POLICY MODEL DIAGNOSTIC")
print("=" * 80)

# ============================================================================
# SCENARIO FROM WEBAPP SCREENSHOT:
# Hero: JJ (pocket jacks)
# Board: 6s 9d 5d As (turn)
# Situation: Hero has overpair on flop, now facing Ace on turn
# Pot: 77, Hero stack: 198, Committed: 22
# Opponent: Lynn (weak read)
# ============================================================================

obvious_bet_scenarios = [
    # PREFLOP SCENARIOS
    {
        'name': 'PREFLOP: AA in position, first to act',
        'street': 'preflop',
        'position': 'BTN',
        'predicted_bucket': 'air',
        'overrides': {
            'hand_equity': 0.85,
            'position_from_button': 0,
            'starting_stack': 100.0,
            'strength_advantage': 0.35,
            'strength_strong': 1,
            'opponent_air_count': 5,
            'heads_up': 0,
            'multiway': 1,
        },
        'expected_action': 'bet',
        'reason': 'Premium hand, should raise for value'
    },
    {
        'name': 'PREFLOP: KK facing limpers',
        'street': 'preflop',
        'position': 'CO',
        'predicted_bucket': 'air',
        'overrides': {
            'hand_equity': 0.82,
            'position_from_button': 1,
            'strength_advantage': 0.32,
            'strength_strong': 1,
            'opponent_air_count': 3,
            'heads_up': 0,
            'multiway': 1,
        },
        'expected_action': 'bet',
        'reason': 'Premium hand vs weak ranges, must raise'
    },
    
    # FLOP SCENARIOS
    {
        'name': 'FLOP: JJ overpair on 6-9-5 rainbow (WEBAPP SCENARIO)',
        'street': 'flop',
        'position': 'BTN',
        'predicted_bucket': 'air',
        'overrides': {
            'hand_equity': 0.78,
            'hole_pair_flag': 1,
            'position_from_button': 0,
            'starting_stack': 198.0,
            'num_players': 2,
            'strength_advantage': 0.28,
            'strength_strong': 1,
            'opponent_strength_mean': 0.3,
            'opponent_air_count': 1,
            'heads_up': 1,
            'board_pair_or_better': 0,
            'board_flush_possible': 0,
            'board_straight_possible': 0,
        },
        'expected_action': 'bet',
        'reason': 'Overpair vs weak opponent, clear value bet'
    },
    {
        'name': 'FLOP: Top set on dry board',
        'street': 'flop',
        'position': 'BTN',
        'predicted_bucket': 'air',
        'overrides': {
            'hand_equity': 0.95,
            'hole_pair_flag': 1,
            'board_pair_or_better': 1,
            'strength_advantage': 0.45,
            'strength_strong': 1,
            'opponent_air_count': 1,
            'heads_up': 1,
        },
        'expected_action': 'bet',
        'reason': 'Monster hand, must bet for value'
    },
    
    # TURN SCENARIOS
    {
        'name': 'TURN: JJ on 6-9-5-A board (WEBAPP - Ace hits)',
        'street': 'turn',
        'position': 'BTN',
        'predicted_bucket': 'air',
        'overrides': {
            'hand_equity': 0.65,
            'hole_pair_flag': 1,
            'position_from_button': 0,
            'starting_stack': 198.0,
            'num_players': 2,
            'strength_advantage': 0.15,
            'strength_strong': 1,
            'opponent_strength_mean': 0.3,
            'opponent_air_count': 1,
            'heads_up': 1,
        },
        'expected_action': 'bet',
        'reason': 'Still likely best hand, can bet thin value'
    },
    {
        'name': 'TURN: Flopped straight, no flush possible',
        'street': 'turn',
        'position': 'BTN',
        'predicted_bucket': 'middle',
        'overrides': {
            'hand_equity': 0.88,
            'board_straight_possible': 1,
            'strength_advantage': 0.38,
            'strength_strong': 1,
            'opponent_middle_count': 1,
            'heads_up': 1,
        },
        'expected_action': 'bet',
        'reason': 'Straight on safe board, bet for value'
    },
    
    # RIVER SCENARIOS
    {
        'name': 'RIVER: Full house vs likely flush',
        'street': 'river',
        'position': 'BTN',
        'predicted_bucket': 'nutted',
        'overrides': {
            'hand_equity': 0.98,
            'hole_pair_flag': 1,
            'board_pair_or_better': 1,
            'board_flush_possible': 1,
            'strength_advantage': 0.48,
            'strength_strong': 1,
            'opponent_nutted_count': 1,
            'heads_up': 1,
        },
        'expected_action': 'bet',
        'reason': 'Boat over flush, max value'
    },
    {
        'name': 'RIVER: Nut straight on paired board',
        'street': 'river',
        'position': 'CO',
        'predicted_bucket': 'middle',
        'overrides': {
            'hand_equity': 0.85,
            'board_pair_or_better': 1,
            'board_straight_possible': 1,
            'strength_advantage': 0.35,
            'strength_strong': 1,
            'opponent_middle_count': 1,
            'heads_up': 1,
        },
        'expected_action': 'bet',
        'reason': 'Strong hand, bet for value'
    },
]

# Run all scenarios
print("\n" + "=" * 80)
print("TESTING OBVIOUS BET SCENARIOS")
print("=" * 80)

results = {'correct': 0, 'wrong': 0, 'missing_model': 0}

for scenario in obvious_bet_scenarios:
    street = scenario['street']
    
    if street not in policy_models:
        print(f"\n   WARNING: {scenario['name']}")
        print(f"   Model not loaded for {street}")
        results['missing_model'] += 1
        continue
    
    model = policy_models[street]
    
    features = create_policy_features(
        position_name=scenario['position'],
        predicted_bucket=scenario['predicted_bucket'],
        **scenario['overrides']
    )
    
    X = pd.DataFrame([features], columns=POLICY_FEATURES)
    
    pred_action_raw = model.predict(X)[0]
    pred_action = get_policy_class_name(pred_action_raw)
    pred_proba = model.predict_proba(X)[0]
    
    # Get classes from model (handle Pipeline)
    classifier = model.named_steps.get('classifier') or model.named_steps.get('model')
    classes = classifier.classes_ if classifier and hasattr(classifier, 'classes_') else [0, 1, 2]
    
    proba_dict = {get_policy_class_name(cls): prob for cls, prob in zip(classes, pred_proba)}
    is_correct = pred_action == scenario['expected_action']
    
    print(f"\n{'='*70}")
    print(f"   {scenario['name']}")
    print(f"{'='*70}")
    print(f"   Street: {street}")
    print(f"   Hand equity: {scenario['overrides'].get('hand_equity', 0.5):.0%}")
    print(f"   Reason to bet: {scenario['reason']}")
    print(f"")
    print(f"   EXPECTED: {scenario['expected_action'].upper()}")
    print(f"   PREDICTED: {pred_action.upper()}")
    print(f"")
    print(f"   Probabilities:")
    for cls in sorted(classes):
        cls_name = get_policy_class_name(cls)
        prob = proba_dict.get(cls_name, 0)
        bar = '#' * int(prob * 30)
        marker = ' <- PREDICTED' if cls_name == pred_action else ''
        expected_marker = ' <- EXPECTED' if cls_name == scenario['expected_action'] else ''
        print(f"      {cls_name:6s}: {prob:5.1%} {bar}{marker}{expected_marker}")
    
    if is_correct:
        print(f"\n   CORRECT")
        results['correct'] += 1
    else:
        print(f"\n   WRONG - Model should recommend {scenario['expected_action']}")
        results['wrong'] += 1

# Summary
print("\n" + "=" * 80)
print("DIAGNOSTIC SUMMARY")
print("=" * 80)
print(f"   Correct predictions: {results['correct']}")
print(f"   Wrong predictions: {results['wrong']}")
print(f"   Missing models: {results['missing_model']}")

if results['wrong'] > 0:
    print(f"\n   MODEL ISSUE DETECTED")
    print(f"   The policy model is not recommending 'bet' for obvious value spots.")
    print(f"   ")
    print(f"   POSSIBLE CAUSES:")
    print(f"   1. Class imbalance - 'call' is majority class in training data")
    print(f"   2. Model learned to be too passive from training data")
    print(f"   3. Features don't capture 'strength' well enough")
    print(f"   ")
    print(f"   CHECK:")
    print(f"   - Action distribution in training data (notebook 09)")
    print(f"   - Whether class_weight='balanced' is being used")
    print(f"   - Feature importance - is hand_equity being used?")
else:
    print(f"\n   Model is correctly identifying obvious bet spots")

In [ ]:
# ============================================================================
# STEP 6: SUMMARY AND RECOMMENDATIONS
# ============================================================================
print("\n" + "=" * 80)
print("VALIDATION SUMMARY")
print("=" * 80)

print("\n✓ Models Loaded:")
print(f"   Opponent models: {list(opponent_models.keys())}")
print(f"   Profit models: {list(profit_models.keys())}")
print(f"   Policy models: {list(policy_models.keys())}")

print("\n✓ Feature Counts (EXACT):")
print(f"   Opponent preflop: {len(OPPONENT_FEATURES_PREFLOP)} features")
print(f"   Opponent postflop: {len(OPPONENT_FEATURES_POSTFLOP)} features")
print(f"   Profit: {len(PROFIT_FEATURES)} features")
print(f"   Policy: {len(POLICY_FEATURES)} features (includes 11 one-hot columns)")

print("\n" + "=" * 80)
print("WEBAPP BUG DIAGNOSIS")
print("=" * 80)

print("""
SCREENSHOT SHOWS: -504.44 BB Expected Profit

POSSIBLE CAUSES:

1. ❌ BB NORMALIZATION MISSING
   - Chip values passed instead of BB values
   - Fix: Divide all chip amounts by big_blind before calling model

2. ❌ FEATURE ORDER MISMATCH  
   - Features passed in wrong order
   - Fix: Use DataFrame with exact column names from PROFIT_FEATURES

3. ❌ MISSING FEATURES
   - Model expects 63 features, webapp sending fewer
   - Fix: Ensure all 63 features are provided

4. ❌ WRONG MODEL VERSION
   - Calling old model that expects different features
   - Fix: Verify using v3 models from Unity Catalog

QUICK DEBUG CHECK IN WEBAPP:
   Print the feature values before calling model:
   
   print(f"pot_size: {features['pot_size']}")  # Should be ~3-50 BB
   print(f"starting_stack: {features['starting_stack']}")  # Should be ~50-300 BB
   print(f"facing_call: {features['facing_call']}")  # Should be ~0-50 BB
   
   If these values are >1000, you're passing chips not BB!
""")

print("\n" + "=" * 80)
print("EXPECTED PREDICTION RANGES")
print("=" * 80)
print("""
   Opponent Model:
   - Output: 'air', 'middle', or 'nutted'
   - Probabilities should sum to 1.0
   
   Profit Model:
   - Typical range: -15 to +15 BB
   - Extreme but valid: -30 to +30 BB
   - ⚠️ If you see -200+ BB: CHECK BB NORMALIZATION!
   
   Policy Model:
   - Output: 'fold', 'call', 'raise_33', 'raise_75', 'shove'
   - Confidence = max_prob - second_prob
""")

# Print the exact features the webapp should be sending
print("\n" + "=" * 80)
print("EXACT FEATURE LIST FOR PROFIT MODEL (63 features)")
print("=" * 80)
for i, f in enumerate(PROFIT_FEATURES):
    print(f"   {i+1:2d}. {f}")

print("\n" + "=" * 80)
print("VALIDATION COMPLETE")
print("=" * 80)